In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, matthews_corrcoef, roc_auc_score, average_precision_score, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import itertools
import xgboost as xgb

# Load Data
data = [
{'sequence': 'KLCEKPSKTWFGNCGNPRHCG', 'value': 32, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'WFGNCGNPRHCG', 'value': 70, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'WEGAVHGACHVRNGKHMC', 'value': 20, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GRCRDDFRCWCTKRC', 'value': 42, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GRCRGFRRRCFCTTHC', 'value': 9, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GRCRGFRRRC', 'value': 9, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RGFRRR', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RECKTESNTFPGICITKPPCRKACISEKFTDGHCSKILRRCLCTKPC', 'value': 1.2, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVNVNAIKKGGKAIGKGFKVISAASTAHDVYEHIKNRRH', 'value': 0.7, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GKIPVKAIKKGGQIIGKALRGINIASTAHDIISQFKPKKKKNH', 'value': 10, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPIGAIKKGGKIIKKGLGVIGAAGTAHEVYSHVKNRH', 'value': 1, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RSGRGECRRQCLRRHEGQPWETQECMRRCRRRG', 'value': 12.7465, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'TDDRCERMCQHYHDRREKKQCMKGCRYGESD', 'value': 5.107, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'CGNFLKRTCICVKK', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFPY', 'value': 465.9138, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GSCGASIAEFNSSQILAKRAPPCRRPRLQNSEDVTHTTLP', 'value': 16.8603, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GSCGAPISKYDFQVLAKRPPPCRRPRLENTEDVTHTTRP', 'value': 5.3509, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'AMAINNWVRVPPCDQVCSRSNPEKDECCRAHGHAFHAHCNGGMNCYRR', 'value': 2.8618, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCRGFRRRCFCTTHC', 'value': 2.25, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHRFKGPCARDSNCATVCLTEGFSGGDCRGFRRRCFCTRPC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RECKTESHRFKGPCITKPPCRKACISEKFTDGHCSKILRRCLCTKPC', 'value': 0.84, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'ALALAI', 'value': 529.677, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'ILIV', 'value': 1566.6001, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KLCEKPSKTWFGNCGNPRRCG', 'value': 38, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KLCEKPSKTWFGNCGNPRACG', 'value': 100, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKAAGPCASDHNCASVCQTERFSGGRCRGFRRRCFCTTHC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCAAFRRRCFCTTHC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCRGAARRCFCTTHC', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCRGFRRACFCTTHC', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GGRCRGFRRRCFCTTHC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'CSGIIKQTCTCYRK', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GLGPNPCRKKCYKRDFLGRCRLNFTCMFG', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'TDDRCERMCQHYHDRREKKQCMKGCRYGESD', 'value': 5.107, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GYYCPFRQDKCHRHCRSFGRKAGYCGNFLKRTCICVKK', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFFCPYNGYCDRHCRKKLRRRGGYCGGRWKLTCICIMN', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFGCPLNQGACHNHCRSIKRRGGYCSGIIKQTCTCYRK', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'AFGCPFDQGTCHSHCRSIRRRGERCSGFAKRTCTCYQK', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFGCPFDQGACHRHCQSIGRRGGYCAGFIKQTCTCYHN', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCENLADKYRGPCFSGCDTHCTTKENAVSGRCRGFRRRCWCTKRC', 'value': 3.9, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCQSQSHRFRGPCLRRSNCANVCRTEGFPGGRCRGFRRRCFCTTHC', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KFCEKPSGTWSGVCGNSGACKDQCIRLEGAKHGSCNYKPPAHRCICYYEC', 'value': 3.2496, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RPDFCLEPPYTGPCKARMIRYFYNAKAGLCQPFVYGGCRAKRNNFKSSEDCMRTCGGA', 'value': 10, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'DDESSKPCCDQCACTKSNPPQCRCSDMRLNSCHSACKSCICALSYPAQCFCVDITDFCYE', 'value': 10, 'type': 'MIC', 'unit': 'µM'}
]

# 转化为DataFrame
df = pd.DataFrame(data)

# 特征提取函数
def aac(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    composition = {aa: sequence.count(aa) for aa in amino_acids}
    composition_total = sum(composition.values())
    normalized_composition = {aa: count / composition_total for aa, count in composition.items()}
    return list(normalized_composition.values())

def three_mer(sequence):
    if len(sequence) < 3:
        return [0] * 64
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    three_mer_combinations = [''.join(comb) for comb in itertools.product(amino_acids, repeat=3)]
    three_mers = [sequence[i:i+3] for i in range(len(sequence) - 2)]
    composition = {three_mer: 0 for three_mer in three_mer_combinations}
    for item in three_mers:
        if item in composition:
            composition[item] += 1
    composition_total = sum(composition.values())
    normalized_composition = {three_mer: count / composition_total for three_mer, count in composition.items()} if composition_total > 0 else {three_mer: 0 for three_mer in composition}
    return list(normalized_composition.values())


def compute_ctd(sequence):
    def compute_composition(sequence):
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        length = len(sequence)
        composition = {aa: sequence.count(aa) / length for aa in amino_acids}
        return list(composition.values())

    def compute_transition(sequence):
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        length = len(sequence)
        transitions = {(aa1, aa2): 0 for aa1 in amino_acids for aa2 in amino_acids}
        for i in range(length - 1):
            aa1, aa2 = sequence[i], sequence[i + 1]
            transitions[(aa1, aa2)] += 1 if (aa1, aa2) in transitions else 0
        normalized_transitions = {pair: count / (length - 1) for pair, count in transitions.items()}
        return list(normalized_transitions.values())

    def compute_distribution(sequence):
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        length = len(sequence)
        distribution = []
        for aa in amino_acids:
            positions = [i / length for i, a in enumerate(sequence) if a == aa]
            if positions:
                distribution.extend([max(min(positions), 0), max(25, min(positions)), max(50, min(positions)), max(100, min(positions)), max(positions)])
            else:
                distribution.extend([0.0] * 5)
        return distribution

    composition = compute_composition(sequence)
    transition = compute_transition(sequence)
    distribution = compute_distribution(sequence)
    return composition + transition + distribution

def pse_aac(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    lambda_ = 10  # Number of correlation factors
    weight = 0.05  # Weight factor
    
    # Compute primary composition
    aac = {aa: sequence.count(aa) for aa in amino_acids}
    total_count = sum(aac.values())
    normalized_aac = [aac[aa] / total_count for aa in amino_acids]
    
    # Compute correlation factors (dummy in this case)
    correlation_factors = [0.1] * lambda_

    # Combine AAC and correlation factors
    for cf in correlation_factors:
        normalized_aac.append(weight * cf / (1 + weight * sum(correlation_factors)))
    
    return normalized_aac

# 计算Dipeptide Composition (DPC)
def dpc(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    dip_combinations = [aa1+aa2 for aa1 in amino_acids for aa2 in amino_acids]
    dpc_composition = {comb: 0 for comb in dip_combinations}
    for i in range(len(sequence) - 1):
        dipeptide = sequence[i:i+2]
        if dipeptide in dpc_composition:
            dpc_composition[dipeptide] += 1
    total_dpc = sum(dpc_composition.values())
    normalized_dpc = {comb: count / total_dpc for comb, count in dpc_composition.items()}
    return list(normalized_dpc.values())

# 计算NCC特征（使用计数的方法）
def ncc(sequence, k=3):
    if len(sequence) < k:
        return [0] * (20 ** k)
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    ncc_combinations = [''.join(comb) for comb in itertools.product(amino_acids, repeat=k)]
    ncc_composition = {comb: 0 for comb in ncc_combinations}
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in ncc_composition:
            ncc_composition[kmer] += 1
    total_ncc = sum(ncc_composition.values())
    normalized_ncc = {comb: count / total_ncc for comb, count in ncc_composition.items()}
    return list(normalized_ncc.values())

df['aac'] = df['sequence'].apply(aac)
df['three_mer'] = df['sequence'].apply(three_mer)
df['ctd'] = df['sequence'].apply(compute_ctd)
df['pse_aac'] = df['sequence'].apply(pse_aac)
df['dpc'] = df['sequence'].apply(dpc)
df['ncc'] = df['sequence'].apply(lambda seq: ncc(seq, 3))

df['features'] = df.apply(lambda row: row['aac'] + row['three_mer'] + row['ctd'] + row['pse_aac'] + row['dpc'] + row['ncc'], axis=1)

# 用MinMaxScaler进行目标值归一化
scaler = MinMaxScaler()
df['normalized_value'] = scaler.fit_transform(df[['value']])

# 准备特征矩阵X和目标向量y
X = np.array(df['features'].tolist())
y = df['normalized_value']

# 定义SimpleNN模型
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 定义评估模型的函数
def evaluate_model(y_true, y_pred, y_true_binary, y_pred_binary, scaler=None):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mcc = matthews_corrcoef(y_true_binary, y_pred_binary)
    r2 = r2_score(y_true, y_pred)
    if scaler:
        y_true_inverse = scaler.inverse_transform(y_true.values.reshape(-1, 1)).flatten()
        y_pred_inverse = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
        mse_inverse = mean_squared_error(y_true_inverse, y_pred_inverse)
        rmse_inverse = np.sqrt(mse_inverse)
    else:
        mse_inverse = np.nan
        rmse_inverse = np.nan
    if len(np.unique(y_true_binary)) > 1:
        roc_auc = roc_auc_score(y_true_binary, y_pred)
        pr_auc = average_precision_score(y_true_binary, y_pred)
    else:
        roc_auc = np.nan
        pr_auc = np.nan
    return mse, rmse, mcc, roc_auc, pr_auc, mse_inverse, rmse_inverse, r2

# 准备保存结果的列表
results = []

# 遍历不同的random_state
for random_state in range(10024):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=random_state)

    # 训练SVR模型
    svr_model = SVR()
    svr_model.fit(X_train, y_train)
    y_pred_svr = svr_model.predict(X_val)

    # 训练随机森林回归模型
    rf_model = RandomForestRegressor()
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_val)

    # 训练PyTorch简单全连接神经网络
    input_dim = X_train.shape[1]
    model = SimpleNN(input_dim)

    # 损失函数和优化器
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # 转换数据为torch tensor
    X_train_torch = torch.tensor(X_train, dtype=torch.float32)
    y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
    X_val_torch = torch.tensor(X_val, dtype=torch.float32)
    y_val_torch = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)

    # 模型训练
    num_epochs = 1000
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_torch)
        loss = criterion(outputs, y_train_torch)
        loss.backward()
        optimizer.step()

    # 评估模型
    model.eval()
    with torch.no_grad():
        y_pred_nn = model(X_val_torch).flatten().numpy()

    # XGBoost模型定义与训练
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    params = {'objective': 'reg:squarederror', 'max_depth': 5, 'eta': 0.1, 'eval_metric': 'rmse'}
    bst = xgb.train(params, dtrain, num_boost_round=1000)

    # 预测
    y_pred_xgb = bst.predict(dval)

    # 计算阈值
    y_min = df['value'].min()
    y_max = df['value'].max()
    threshold_original = 10
    threshold_normalized = (threshold_original - y_min) / (y_max - y_min)

    # 二值化标签
    y_val_binary = (y_val >= threshold_normalized).astype(int)
    y_pred_svr_binary = (y_pred_svr >= threshold_normalized).astype(int)
    y_pred_rf_binary = (y_pred_rf >= threshold_normalized).astype(int)
    y_pred_nn_binary = (y_pred_nn >= threshold_normalized).astype(int)
    y_pred_xgb_binary = (y_pred_xgb >= threshold_normalized).astype(int)

    # 评估每个模型
    svr_results = evaluate_model(y_val, y_pred_svr, y_val_binary, y_pred_svr_binary, scaler)
    rf_results = evaluate_model(y_val, y_pred_rf, y_val_binary, y_pred_rf_binary, scaler)
    nn_results = evaluate_model(y_val, y_pred_nn, y_val_binary, y_pred_nn_binary, scaler)
    xgb_results = evaluate_model(y_val, y_pred_xgb, y_val_binary, y_pred_xgb_binary, scaler)

    # 保存结果
    results.append([random_state, 'SVR'] + list(svr_results))
    results.append([random_state, 'RandomForest'] + list(rf_results))
    results.append([random_state, 'SimpleNN'] + list(nn_results))
    results.append([random_state, 'XGBoost'] + list(xgb_results))

# 将结果转换为DataFrame
results_df = pd.DataFrame(results, columns=['random_state', 'model', 'mse', 'rmse', 'mcc', 'roc_auc', 'pr_auc', 'mse_inverse', 'rmse_inverse', 'r2'])

# 保存结果为Excel文件
results_df.to_excel('model_results.xlsx', index=False)

print("结果已保存到 model_results.xlsx 文件中")

结果已保存到 model_results.xlsx 文件中


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, matthews_corrcoef, roc_auc_score, average_precision_score, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import itertools
import xgboost as xgb

# Load Data
data = [
{'sequence': 'KLCEKPSKTWFGNCGNPRHCG', 'value': 32, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'WFGNCGNPRHCG', 'value': 70, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'WEGAVHGACHVRNGKHMC', 'value': 20, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GRCRDDFRCWCTKRC', 'value': 42, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GRCRGFRRRCFCTTHC', 'value': 9, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GRCRGFRRRC', 'value': 9, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RGFRRR', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RECKTESNTFPGICITKPPCRKACISEKFTDGHCSKILRRCLCTKPC', 'value': 1.2, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVNVNAIKKGGKAIGKGFKVISAASTAHDVYEHIKNRRH', 'value': 0.7, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GKIPVKAIKKGGQIIGKALRGINIASTAHDIISQFKPKKKKNH', 'value': 10, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPIGAIKKGGKIIKKGLGVIGAAGTAHEVYSHVKNRH', 'value': 1, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPIGAIKKGGKIIKKGLGVIGAAGTAHEVYSHVKNRH', 'value': 1, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPIGAIKKGGKIIKKGLGVIGAAGTAHEVYSHVKNRH', 'value': 10, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPVGAIKKGGKAIKTGLGVVGAAGTAHEVYSHIRNRH', 'value': 100, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPVGAIKKGGKAIKTGLGVVGAAGTAHEVYSHIRNRH', 'value': 10, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KVPVGAIKKGGKAIKTGLGVVGAAGTAHEVYSHIRNRH', 'value': 9.9008, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RSGRGECRRQCLRRHEGQPWETQECMRRCRRRG', 'value': 12.7465, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RSGRGECRRQCLRRHEGQPWETQECMRRCRRRG', 'value': 4.78, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'TDDRCERMCQHYHDRREKKQCMKGCRYGESD', 'value': 5.107, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'CGNFLKRTCICVKK', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFPY', 'value': 465.9138, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GSCGASIAEFNSSQILAKRAPPCRRPRLQNSEDVTHTTLP', 'value': 16.8603, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GSCGAPISKYDFQVLAKRPPPCRRPRLENTEDVTHTTRP', 'value': 5.3509, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'AMAINNWVRVPPCDQVCSRSNPEKDECCRAHGHAFHAHCNGGMNCYRR', 'value': 2.8618, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCRGFRRRCFCTTHC', 'value': 2.25, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHRFKGPCARDSNCATVCLTEGFSGGDCRGFRRRCFCTRPC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RECKTESHRFKGPCITKPPCRKACISEKFTDGHCSKILRRCLCTKPC', 'value': 0.84, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'ALALAI', 'value': 529.677, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'ILIV', 'value': 1566.6001, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KLCEKPSKTWFGNCGNPRRCG', 'value': 38, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KLCEKPSKTWFGNCGNPRACG', 'value': 100, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKAAGPCASDHNCASVCQTERFSGGRCRGFRRRCFCTTHC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCAAFRRRCFCTTHC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCRGAARRCFCTTHC', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCESQSHKFKGPCASDHNCASVCQTERFSGGRCRGFRRACFCTTHC', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GGRCRGFRRRCFCTTHC', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'CSGIIKQTCTCYRK', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GLGPNPCRKKCYKRDFLGRCRLNFTCMFG', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'TDDRCERMCQHYHDRREKKQCMKGCRYGESD', 'value': 5.107, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GYYCPFRQDKCHRHCRSFGRKAGYCGNFLKRTCICVKK', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFFCPYNGYCDRHCRKKLRRRGGYCGGRWKLTCICIMN', 'value': 12, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFGCPLNQGACHNHCRSIKRRGGYCSGIIKQTCTCYRK', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'AFGCPFDQGTCHSHCRSIRRRGERCSGFAKRTCTCYQK', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'GFGCPFDQGACHRHCQSIGRRGGYCAGFIKQTCTCYHN', 'value': 6, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCENLADKYRGPCFSGCDTHCTTKENAVSGRCRGFRRRCWCTKRC', 'value': 3.9, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RTCQSQSHRFRGPCLRRSNCANVCRTEGFPGGRCRGFRRRCFCTTHC', 'value': 3, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'KFCEKPSGTWSGVCGNSGACKDQCIRLEGAKHGSCNYKPPAHRCICYYEC', 'value': 3.2496, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'RPDFCLEPPYTGPCKARMIRYFYNAKAGLCQPFVYGGCRAKRNNFKSSEDCMRTCGGA', 'value': 10, 'type': 'MIC', 'unit': 'µM'},
{'sequence': 'DDESSKPCCDQCACTKSNPPQCRCSDMRLNSCHSACKSCICALSYPAQCFCVDITDFCYE', 'value': 10, 'type': 'MIC', 'unit': 'µM'}
]

# 转化为DataFrame
df = pd.DataFrame(data)

# 特征提取函数
def aac(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    composition = {aa: sequence.count(aa) for aa in amino_acids}
    composition_total = sum(composition.values())
    normalized_composition = {aa: count / composition_total for aa, count in composition.items()}
    return list(normalized_composition.values())

def three_mer(sequence):
    if len(sequence) < 3:
        return [0] * 64
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    three_mer_combinations = [''.join(comb) for comb in itertools.product(amino_acids, repeat=3)]
    three_mers = [sequence[i:i+3] for i in range(len(sequence) - 2)]
    composition = {three_mer: 0 for three_mer in three_mer_combinations}
    for item in three_mers:
        if item in composition:
            composition[item] += 1
    composition_total = sum(composition.values())
    normalized_composition = {three_mer: count / composition_total for three_mer, count in composition.items()} if composition_total > 0 else {three_mer: 0 for three_mer in composition}
    return list(normalized_composition.values())


def compute_ctd(sequence):
    def compute_composition(sequence):
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        length = len(sequence)
        composition = {aa: sequence.count(aa) / length for aa in amino_acids}
        return list(composition.values())

    def compute_transition(sequence):
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        length = len(sequence)
        transitions = {(aa1, aa2): 0 for aa1 in amino_acids for aa2 in amino_acids}
        for i in range(length - 1):
            aa1, aa2 = sequence[i], sequence[i + 1]
            transitions[(aa1, aa2)] += 1 if (aa1, aa2) in transitions else 0
        normalized_transitions = {pair: count / (length - 1) for pair, count in transitions.items()}
        return list(normalized_transitions.values())

    def compute_distribution(sequence):
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        length = len(sequence)
        distribution = []
        for aa in amino_acids:
            positions = [i / length for i, a in enumerate(sequence) if a == aa]
            if positions:
                distribution.extend([max(min(positions), 0), max(25, min(positions)), max(50, min(positions)), max(100, min(positions)), max(positions)])
            else:
                distribution.extend([0.0] * 5)
        return distribution

    composition = compute_composition(sequence)
    transition = compute_transition(sequence)
    distribution = compute_distribution(sequence)
    return composition + transition + distribution

def pse_aac(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    lambda_ = 10  # Number of correlation factors
    weight = 0.05  # Weight factor
    
    # Compute primary composition
    aac = {aa: sequence.count(aa) for aa in amino_acids}
    total_count = sum(aac.values())
    normalized_aac = [aac[aa] / total_count for aa in amino_acids]
    
    # Compute correlation factors (dummy in this case)
    correlation_factors = [0.1] * lambda_

    # Combine AAC and correlation factors
    for cf in correlation_factors:
        normalized_aac.append(weight * cf / (1 + weight * sum(correlation_factors)))
    
    return normalized_aac

# 计算Dipeptide Composition (DPC)
def dpc(sequence):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    dip_combinations = [aa1+aa2 for aa1 in amino_acids for aa2 in amino_acids]
    dpc_composition = {comb: 0 for comb in dip_combinations}
    for i in range(len(sequence) - 1):
        dipeptide = sequence[i:i+2]
        if dipeptide in dpc_composition:
            dpc_composition[dipeptide] += 1
    total_dpc = sum(dpc_composition.values())
    normalized_dpc = {comb: count / total_dpc for comb, count in dpc_composition.items()}
    return list(normalized_dpc.values())

# 计算NCC特征（使用计数的方法）
def ncc(sequence, k=3):
    if len(sequence) < k:
        return [0] * (20 ** k)
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    ncc_combinations = [''.join(comb) for comb in itertools.product(amino_acids, repeat=k)]
    ncc_composition = {comb: 0 for comb in ncc_combinations}
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in ncc_composition:
            ncc_composition[kmer] += 1
    total_ncc = sum(ncc_composition.values())
    normalized_ncc = {comb: count / total_ncc for comb, count in ncc_composition.items()}
    return list(normalized_ncc.values())

df['aac'] = df['sequence'].apply(aac)
df['three_mer'] = df['sequence'].apply(three_mer)
df['ctd'] = df['sequence'].apply(compute_ctd)
df['pse_aac'] = df['sequence'].apply(pse_aac)
df['dpc'] = df['sequence'].apply(dpc)
df['ncc'] = df['sequence'].apply(lambda seq: ncc(seq, 3))

df['features'] = df.apply(lambda row: row['aac'] + row['three_mer'] + row['ctd'] + row['pse_aac'] + row['dpc'] + row['ncc'], axis=1)

# 用MinMaxScaler进行目标值归一化
scaler = MinMaxScaler()
df['normalized_value'] = scaler.fit_transform(df[['value']])

# 准备特征矩阵X和目标向量y
X = np.array(df['features'].tolist())
y = df['normalized_value']

# 分割数据集
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=2157)

# 训练XGBoost模型
dtrain = xgb.DMatrix(X_train, label=y_train)
params = {'objective': 'reg:squarederror', 'max_depth': 5, 'eta': 0.1, 'eval_metric': 'rmse'}
bst = xgb.train(params, dtrain, num_boost_round=1000)

# 准备新序列的特征
new_sequence = 'KGPPRR'
new_features = aac(new_sequence) + three_mer(new_sequence) + compute_ctd(new_sequence) + \
               pse_aac(new_sequence) + dpc(new_sequence) + ncc(new_sequence, 3)

# 将新特征转换为XGBoost可以使用的格式
new_features_matrix = xgb.DMatrix([new_features])

# 使用训练好的模型进行预测
predicted_normalized_value = bst.predict(new_features_matrix)[0]

# 反归一化预测结果
scaler = MinMaxScaler()
scaler.fit(df[['value']])
predicted_value = scaler.inverse_transform([[predicted_normalized_value]])[0][0]

print(f"序列 'KGPPRR' 的预测抗菌肽活性 (MIC) 为: {predicted_value:.2f} µM")

序列 'KGPPRR' 的预测抗菌肽活性 (MIC) 为: 10.25 µM
